In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [20]:
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score

In [21]:
df = pd.read_csv(
    "../data/processed/final_training_data.csv"
)

X = df.drop(
    columns=["is_high_risk"]
)

y = df["is_high_risk"]

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

print(X_train.shape)
print(X_test.shape)

(76529, 34)
(19133, 34)


In [22]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

In [23]:
grid_search = GridSearchCV(
    RandomForestClassifier(
        random_state=42
    ),
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 12 candidates, totalling 36 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='roc_auc', verbose=2)

In [24]:
print(grid_search.best_params_)

{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [25]:
hasattr(grid_search, "best_params_")

True

In [26]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest ROC-AUC:")
print(grid_search.best_score_)

Best Parameters:
{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}

Best ROC-AUC:
0.9981601688619971


In [27]:
best_model = grid_search.best_estimator_

with mlflow.start_run(
    run_name="RandomForest_Tuned"
):

    mlflow.log_params(
        grid_search.best_params_
    )

    y_prob = best_model.predict_proba(
        X_test
    )[:, 1]

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    mlflow.log_metric(
        "roc_auc",
        roc_auc
    )

    mlflow.sklearn.log_model(
        best_model,
        "RandomForest_Tuned"
    )

print("Model logged successfully")

Model logged successfully


c:\credit-risk-model\venv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\credit-risk-model\venv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [28]:
print(type(best_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
